# Active Inference Epidemic — Exploration Notebook

This notebook runs experiments and generates all figures for the repository.

**Structure:**
1. Single episode — belief tracking visualisation
2. Epistemic vs. pragmatic G decomposition
3. Phase portrait
4. Ablation: AI agent vs. greedy vs. random
5. Sensitivity analysis — effect of temperature τ

In [ ]:
import sys
sys.path.append('../src')

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from environment import EpiParams, DEFAULT_PARAMS
from simulate import run_episode, run_ablation
from plot import (
    fig_belief_tracking,
    fig_action_decomp,
    fig_phase_portrait,
    fig_ablation,
)

print(f'JAX version: {jax.__version__}')
print(f'Devices: {jax.devices()}')

## 1. Single Episode

In [ ]:
results = run_episode(
    n_steps=120,
    params=DEFAULT_PARAMS,
    I_init=0.01,
    temperature=1.0,
    seed=42,
    verbose=True,
)

In [ ]:
fig = fig_belief_tracking(results, save_path='../results/belief_tracking.png')
plt.show()

## 2. G Decomposition — Epistemic vs Pragmatic

In [ ]:
fig = fig_action_decomp(results, save_path='../results/action_decomp.png')
plt.show()

## 3. Phase Portrait

In [ ]:
fig = fig_phase_portrait(results, save_path='../results/phase_portrait.png')
plt.show()

## 4. Ablation Study

In [ ]:
ablation = run_ablation(n_steps=120, params=DEFAULT_PARAMS, seed=42)

for name, res in ablation.items():
    I_peak = res['true_states'][:, 1].max()
    print(f'{name:25s}  peak I = {I_peak:.3%}')

In [ ]:
fig = fig_ablation(ablation, save_path='../results/ablation.png')
plt.show()

## 5. Sensitivity: Temperature τ

Temperature controls the exploration–exploitation balance at the *policy* level.
Low τ → greedy (always pick lowest G). High τ → uniform random.

In [ ]:
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]
peak_infections = []

for tau in temperatures:
    r = run_episode(n_steps=120, temperature=tau, seed=42)
    I_peak = r['true_states'][:, 1].max()
    peak_infections.append(float(I_peak))
    print(f'τ={tau:.1f}  →  peak I = {I_peak:.3%}')

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0F1117')
ax.set_facecolor('#0F1117')
for spine in ax.spines.values(): spine.set_color('#333')

ax.plot(temperatures, peak_infections, 'o-', color='#00D4FF', lw=2)
ax.set_xlabel('Temperature τ', color='white')
ax.set_ylabel('Peak infected fraction', color='white')
ax.set_title('Effect of Policy Temperature on Epidemic Control', color='white')
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('../results/temperature_sensitivity.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## Free Energy Convergence

Verify that the variational update actually converges — F should decrease monotonically.

In [ ]:
F_trace = results['free_energies']

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0F1117')
ax.set_facecolor('#0F1117')
for spine in ax.spines.values(): spine.set_color('#333')

ax.plot(F_trace, color='#F5A623', lw=1.5)
ax.set_xlabel('Time step', color='white')
ax.set_ylabel('Variational Free Energy F', color='white')
ax.set_title('Free Energy over Episode (after variational update)', color='white')
ax.tick_params(colors='white')
plt.tight_layout()
plt.show()